# **SpaceX Falcon 9 First Stage Landing Prediction**
## **Lab 1: Collecting the Data via SpaceX REST API**

In this capstone project, we predict whether the Falcon 9 first stage will land successfully. SpaceX advertises Falcon 9 rocket launches on its website with a cost of 62 million dollars; other providers cost upward of 165 million dollars each, much of the savings is because SpaceX can reuse the first stage. Therefore, if we can determine if the first stage will land, we can determine the cost of a launch. This information can be used if an alternate company wants to bid against SpaceX for a rocket launch.

In this notebook, we make a `GET` request to the SpaceX REST API, enrich the launch records using relational endpoints (`/rockets`, `/launchpads`, `/payloads`, `/cores`), clean the dataset to isolate single-core Falcon 9 launches, impute missing `PayloadMass` values, and export the final dataset to `dataset_part_1.csv`.

### **Objectives**
* Request data from the SpaceX API (`https://api.spacexdata.com/v4/launches/past`).
* Normalize the JSON response into a Pandas DataFrame.
* Enrich the dataset using helper functions for rockets, launchpads, payloads, and cores.
* Filter the data to include only single-core, single-payload Falcon 9 launches prior to November 13, 2020.
* Handle missing values (`PayloadMass` mean imputation; retain `LandingPad` `None` values).
* Export the cleaned dataset to `dataset_part_1.csv`.

### **1. Import Libraries and Define Auxiliary Functions**

In [1]:
import requests
import pandas as pd
import numpy as np
import datetime

# Setting display options to show all columns and full column width
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Takes the dataset and uses the rocket column to call the API and append the data to the list
def getBoosterVersion(data):
    for x in data['rocket']:
        if x:
            response = requests.get("https://api.spacexdata.com/v4/rockets/" + str(x)).json()
            BoosterVersion.append(response['name'])

# Takes the dataset and uses the launchpad column to call the API and append the data to the list
def getLaunchSite(data):
    for x in data['launchpad']:
        if x:
            response = requests.get("https://api.spacexdata.com/v4/launchpads/" + str(x)).json()
            Longitude.append(response['longitude'])
            Latitude.append(response['latitude'])
            LaunchSite.append(response['name'])

# Takes the dataset and uses the payloads column to call the API and append the data to the list
def getPayloadData(data):
    for load in data['payloads']:
        if load:
            response = requests.get("https://api.spacexdata.com/v4/payloads/" + str(load)).json()
            PayloadMass.append(response['mass_kg'])
            Orbit.append(response['orbit'])

# Takes the dataset and uses the cores column to call the API and append the data to the list
def getCoreData(data):
    for core in data['cores']:
        if core['core'] != None:
            response = requests.get("https://api.spacexdata.com/v4/cores/" + core['core']).json()
            Block.append(response['block'])
            ReusedCount.append(response['reuse_count'])
            Serial.append(response['serial'])
        else:
            Block.append(None)
            ReusedCount.append(None)
            Serial.append(None)
        Outcome.append(str(core['landing_success']) + ' ' + str(core['landing_type']))
        Flights.append(core['flight'])
        GridFins.append(core['gridfins'])
        Reused.append(core['reused'])
        Legs.append(core['legs'])
        LandingPad.append(core['landpad'])

print("Auxiliary API lookup functions defined.")

Auxiliary API lookup functions defined.


### **2. Request and Parse SpaceX Launch Data**

In [3]:
# Static JSON URL provided for reproducible IBM Capstone dataset
static_json_url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json'

response = requests.get(static_json_url)
print("HTTP Status Code:", response.status_code)

HTTP Status Code: 200


In [4]:
# Decode JSON response and normalize into a flat Pandas DataFrame
data = pd.json_normalize(response.json())
print("Raw DataFrame Shape:", data.shape)
data[['flight_number', 'name', 'date_utc', 'rocket', 'launchpad', 'success']].head(5)

Raw DataFrame Shape: (107, 42)


   flight_number         name                  date_utc                    rocket                 launchpad success
0              1    FalconSat  2006-03-24T22:30:00.000Z  5e9d0d95eda69955f709d1eb  5e9e4502f5090995de566f86   False
1              2      DemoSat  2007-03-21T01:10:00.000Z  5e9d0d95eda69955f709d1eb  5e9e4502f5090995de566f86   False
2              3  Trailblazer  2008-08-03T03:34:00.000Z  5e9d0d95eda69955f709d1eb  5e9e4502f5090995de566f86   False
3              4       RatSat  2008-09-28T23:15:00.000Z  5e9d0d95eda69955f709d1eb  5e9e4502f5090995de566f86    True
4              5     RazakSat  2009-07-13T03:35:00.000Z  5e9d0d95eda69955f709d1eb  5e9e4502f5090995de566f86    True

In [5]:
# Subset the DataFrame to retain only required columns
data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']]

# Remove rows with multiple cores (Falcon Heavy) and multiple payloads
data = data[data['cores'].map(len) == 1]
data = data[data['payloads'].map(len) == 1]

# Extract single values from lists of size 1
data['cores'] = data['cores'].map(lambda x: x[0])
data['payloads'] = data['payloads'].map(lambda x: x[0])

# Convert date_utc to datetime.date and filter launches up to 2020-11-13
data['date'] = pd.to_datetime(data['date_utc']).dt.date
data = data[data['date'] <= datetime.date(2020, 11, 13)]

print("Filtered DataFrame Shape:", data.shape)

Filtered DataFrame Shape: (94, 7)


### **3. Enrich Dataset via Secondary SpaceX API Endpoints**

In [6]:
# Initialize global lists to store extracted attributes
BoosterVersion = []
PayloadMass = []
Orbit = []
LaunchSite = []
Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []
Block = []
ReusedCount = []
Serial = []
Longitude = []
Latitude = []

# Populate lists using API lookup functions
getBoosterVersion(data)
getLaunchSite(data)
getPayloadData(data)
getCoreData(data)

print(f"Extracted {len(BoosterVersion)} BoosterVersion records.")
print(f"Sample BoosterVersions: {BoosterVersion[:5]}")

Extracted 94 BoosterVersion records.
Sample BoosterVersions: ['Falcon 1', 'Falcon 1', 'Falcon 1', 'Falcon 1', 'Falcon 9']


In [7]:
# Combine extracted lists into a structured dictionary
launch_dict = {
    'FlightNumber': list(data['flight_number']),
    'Date': list(data['date']),
    'BoosterVersion': BoosterVersion,
    'PayloadMass': PayloadMass,
    'Orbit': Orbit,
    'LaunchSite': LaunchSite,
    'Outcome': Outcome,
    'Flights': Flights,
    'GridFins': GridFins,
    'Reused': Reused,
    'Legs': Legs,
    'LandingPad': LandingPad,
    'Block': Block,
    'ReusedCount': ReusedCount,
    'Serial': Serial,
    'Longitude': Longitude,
    'Latitude': Latitude
}

df = pd.DataFrame(launch_dict)
df.head(5)

   FlightNumber        Date BoosterVersion  PayloadMass Orbit       LaunchSite    Outcome  Flights  GridFins  Reused   Legs LandingPad  Block  ReusedCount    Serial   Longitude   Latitude
0             1  2006-03-24       Falcon 1         20.0   LEO  Kwajalein Atoll  None None        1     False   False  False       None    NaN            0  Merlin1A  167.743129   9.047721
1             2  2007-03-21       Falcon 1          NaN   LEO  Kwajalein Atoll  None None        1     False   False  False       None    NaN            0  Merlin2A  167.743129   9.047721
2             4  2008-09-28       Falcon 1        165.0   LEO  Kwajalein Atoll  None None        1     False   False  False       None    NaN            0  Merlin2C  167.743129   9.047721
3             5  2009-07-13       Falcon 1        200.0   LEO  Kwajalein Atoll  None None        1     False   False  False       None    NaN            0  Merlin3C  167.743129   9.047721
4             6  2010-06-04       Falcon 9          NaN   LE

### **4. Filter the DataFrame to Only Include Falcon 9 Launches**

In [8]:
# Remove Falcon 1 launches to keep only Falcon 9
data_falcon9 = df[df['BoosterVersion'] != 'Falcon 1'].copy()

# Reset FlightNumber sequentially from 1 to 90
data_falcon9.loc[:, 'FlightNumber'] = list(range(1, data_falcon9.shape[0] + 1))

print("Falcon 9 Dataset Shape:", data_falcon9.shape)
data_falcon9.head(5)

Falcon 9 Dataset Shape: (90, 17)


   FlightNumber        Date BoosterVersion  PayloadMass Orbit    LaunchSite    Outcome  Flights  GridFins  Reused   Legs LandingPad  Block  ReusedCount Serial   Longitude   Latitude
4             1  2010-06-04       Falcon 9          NaN   LEO  CCSFS SLC 40  None None        1     False   False  False       None    1.0            0  B0003  -80.577366  28.561857
5             2  2012-05-22       Falcon 9        525.0   LEO  CCSFS SLC 40  None None        1     False   False  False       None    1.0            0  B0005  -80.577366  28.561857
6             3  2013-03-01       Falcon 9        677.0   ISS  CCSFS SLC 40  None None        1     False   False  False       None    1.0            0  B0007  -80.577366  28.561857
7             4  2013-09-29       Falcon 9        500.0    PO   VAFB SLC 4E  False Ocean       1     False   False  False       None    1.0            0  B1003 -120.610829  34.632093
8             5  2013-12-03       Falcon 9       3170.0   GTO  CCSFS SLC 40  None None   

### **5. Data Wrangling: Handling Missing Values & Exporting CSV**

In [9]:
# Check missing values across all columns before imputation
print("Missing values BEFORE imputation:")
print(data_falcon9.isnull().sum())

Missing values BEFORE imputation:
FlightNumber       0
Date              0
BoosterVersion    0
PayloadMass       5
Orbit             0
LaunchSite        0
Outcome           0
Flights           0
GridFins          0
Reused            0
Legs              0
LandingPad       26
Block             0
ReusedCount       0
Serial            0
Longitude         0
Latitude          0
dtype: int64


In [10]:
# Calculate the mean value of PayloadMass and replace NaN values
payload_mean = data_falcon9['PayloadMass'].mean()
print(f"Mean PayloadMass: {payload_mean:.2f} kg")

data_falcon9['PayloadMass'] = data_falcon9['PayloadMass'].replace(np.nan, payload_mean)

print("\nMissing values AFTER imputation:")
print(data_falcon9.isnull().sum())

Mean PayloadMass: 6123.55 kg

Missing values AFTER imputation:
FlightNumber       0
Date              0
BoosterVersion    0
PayloadMass       0
Orbit             0
LaunchSite        0
Outcome           0
Flights           0
GridFins          0
Reused            0
Legs              0
LandingPad       26
Block             0
ReusedCount       0
Serial            0
Longitude         0
Latitude          0
dtype: int64


In [11]:
# Export cleaned dataset to CSV for downstream EDA and modeling labs
data_falcon9.to_csv('dataset_part_1.csv', index=False)
print("Successfully exported dataset_part_1.csv (90 rows, 17 columns).")

Successfully exported dataset_part_1.csv (90 rows, 17 columns).
